In [27]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("../data/raw/Sample_Superstore.csv")

df_clean = df.copy()

print("Original Dataset Shape:", df.shape)
print("Working Dataset Shape:", df_clean.shape)

df_clean.head()

Original Dataset Shape: (9994, 22)
Working Dataset Shape: (9994, 22)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Unique id
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,1
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,0
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,1
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,1
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,0


In [28]:
print("Column Names:")
print(df_clean.columns.tolist())

print("\nDataset Information:")
df_clean.info()

Column Names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Unique id']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    999

In [29]:
df_clean = df_clean.drop(columns=["Unique id"], errors="ignore")

print("Shape after dropping unnecessary column:")
print(df_clean.shape)

Shape after dropping unnecessary column:
(9994, 21)


In [30]:
# Convert dates
df_clean["Order Date"] = pd.to_datetime(df_clean["Order Date"])

df_clean["Ship Date"] = pd.to_datetime(df_clean["Ship Date"])

# Fix Postal Code
df_clean["Postal Code"] = (
    df_clean["Postal Code"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

print(df_clean.dtypes)

Row ID                    int64
Order ID                    str
Order Date       datetime64[us]
Ship Date        datetime64[us]
Ship Mode                   str
Customer ID                 str
Customer Name               str
Segment                     str
Country                     str
City                        str
State                       str
Postal Code              string
Region                      str
Product ID                  str
Category                    str
Sub-Category                str
Product Name                str
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object


In [31]:
# Exclude Row ID while checking duplicates
cols = [c for c in df_clean.columns if c != "Row ID"]

duplicates = df_clean[
    df_clean.duplicated(subset=cols, keep=False)
]

print("Duplicate rows found:")
display(duplicates)

print("Total duplicate records:", 
      df_clean.duplicated(subset=cols).sum())

Duplicate rows found:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
3405,3406,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588
3406,3407,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588


Total duplicate records: 1


In [32]:
df_clean = df_clean.drop_duplicates(
    subset=cols,
    keep="first"
)

print("Shape after removing duplicates:")
print(df_clean.shape)

Shape after removing duplicates:
(9993, 21)


In [33]:
invalid_dates = (
    df_clean["Ship Date"] < df_clean["Order Date"]
).sum()

print("Invalid shipping dates:", invalid_dates)

Invalid shipping dates: 0


In [34]:
categories = [
    "Segment",
    "Region",
    "Category",
    "Sub-Category",
    "Ship Mode"
]

for col in categories:
    print("\n", col)
    print(df_clean[col].unique())


 Segment
<ArrowStringArray>
['Consumer', 'Corporate', 'Home Office']
Length: 3, dtype: str

 Region
<ArrowStringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: str

 Category
<ArrowStringArray>
['Furniture', 'Office Supplies', 'Technology']
Length: 3, dtype: str

 Sub-Category
<ArrowStringArray>
[  'Bookcases',      'Chairs',      'Labels',      'Tables',     'Storage',
 'Furnishings',         'Art',      'Phones',     'Binders',  'Appliances',
       'Paper', 'Accessories',   'Envelopes',   'Fasteners',    'Supplies',
    'Machines',     'Copiers']
Length: 17, dtype: str

 Ship Mode
<ArrowStringArray>
['Second Class', 'Standard Class', 'First Class', 'Same Day']
Length: 4, dtype: str


In [35]:
names_per_id = (
    df_clean.groupby("Product ID")["Product Name"]
    .nunique()
)

problem_ids = names_per_id[names_per_id > 1].index

print("Product IDs with multiple names:", len(problem_ids))

# Display examples
display(
    df_clean[df_clean["Product ID"].isin(problem_ids[:2])]
    .groupby(["Product ID", "Product Name"])
    .size()
)

Product IDs with multiple names: 32


Product ID       Product Name                                    
FUR-BO-10002213  DMI Eclipse Executive Suite Bookcases                6
                 Sauder Forest Hills Library, Woodland Oak Finish     4
FUR-CH-10001146  Global Task Chair, Black                             5
                 Global Value Mid-Back Manager's Chair, Gray         10
dtype: int64

In [36]:
# Extract year
df_clean["Order Year"] = df_clean["Order Date"].dt.year

# Extract month
df_clean["Order Month"] = df_clean["Order Date"].dt.month

# Calculate profit margin
df_clean["Profit Margin %"] = (
    df_clean["Profit"] /
    df_clean["Sales"].replace(0, np.nan) * 100
).round(2)

# Calculate shipping days
df_clean["Shipping Days"] = (
    df_clean["Ship Date"] - df_clean["Order Date"]
).dt.days

df_clean.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Profit Margin %,Shipping Days
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,16.00,3
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,30.00,3
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,47.00,4
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,-40.00,7
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,11.25,7


In [37]:
print("Missing Values:")
print(df_clean.isnull().sum())

print("\nTotal Missing Values:")
print(df_clean.isnull().sum().sum())

Missing Values:
Row ID             0
Order ID           0
Order Date         0
Ship Date          0
Ship Mode          0
Customer ID        0
Customer Name      0
Segment            0
Country            0
City               0
State              0
Postal Code        0
Region             0
Product ID         0
Category           0
Sub-Category       0
Product Name       0
Sales              0
Quantity           0
Discount           0
Profit             0
Order Year         0
Order Month        0
Profit Margin %    0
Shipping Days      0
dtype: int64

Total Missing Values:
0


In [38]:
print("Final Rows:", df_clean.shape[0])
print("Final Columns:", df_clean.shape[1])

print("\nFinal Dataset:")
display(df_clean.head())

Final Rows: 9993
Final Columns: 25

Final Dataset:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Profit Margin %,Shipping Days
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,16.00,3
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,30.00,3
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,47.00,4
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,-40.00,7
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,11.25,7


In [39]:
os.makedirs("../data/processed", exist_ok=True)

df_clean.to_csv(
    "../data/processed/superstore_clean.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [40]:
df_check = pd.read_csv(
    "../data/processed/superstore_clean.csv"
)

print("Saved Dataset Shape:", df_check.shape)

df_check.head()

Saved Dataset Shape: (9993, 25)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Profit Margin %,Shipping Days
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,16.00,3
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,30.00,3
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,47.00,4
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,-40.00,7
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,11.25,7


In [41]:
print("Total Sales:", df_clean["Sales"].sum())

print("Total Profit:", df_clean["Profit"].sum())

print("Total Orders:", df_clean["Order ID"].nunique())

print("Total Products:", df_clean["Product Name"].nunique())

print("Average Shipping Days:", 
      df_clean["Shipping Days"].mean())

Total Sales: 2296919.4883000003
Total Profit: 286409.0805
Total Orders: 5009
Total Products: 1850
Average Shipping Days: 3.9581707195036526


In [42]:
print("Final Shape:", df_clean.shape)

print("Total Sales:", round(df_clean["Sales"].sum(), 2))

print("Total Profit:", round(df_clean["Profit"].sum(), 2))

print("Duplicate Rows:", df_clean.duplicated().sum())

print("Missing Values:", df_clean.isnull().sum().sum())

print("Product IDs with multiple names:", len(problem_ids))

Final Shape: (9993, 25)
Total Sales: 2296919.49
Total Profit: 286409.08
Duplicate Rows: 0
Missing Values: 0
Product IDs with multiple names: 32


## Data Cleaning Summary

- Original dataset: 9,994 rows and 22 columns.
- Cleaned dataset: 9,993 rows and 25 columns.
- Removed unnecessary `Unique id` column.
- Converted Order Date and Ship Date into datetime format.
- Corrected Postal Code format to five-digit text.
- Identified and removed one duplicate record.
- Identified 32 Product IDs associated with multiple product names.
- Decided to use Product Name for product-level analysis.
- Checked date consistency and categorical columns.
- Created Order Year, Order Month, Profit Margin %, and Shipping Days.
- Saved the cleaned dataset in `data/processed/`.